In [1]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Marilyn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge

In [3]:
data=pd.read_csv('Walmart_Store_sales.csv')

In [4]:
data.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    str    
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), str(1)
memory usage: 10.8 KB


In [6]:
data.describe(include='all')

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,07-01-2011,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [7]:
(data.isnull().sum()/data.shape[0]*100).sort_values(ascending=False)

Date            12.000000
Temperature     12.000000
Unemployment    10.000000
Weekly_Sales     9.333333
Fuel_Price       9.333333
Holiday_Flag     8.000000
CPI              8.000000
Store            0.000000
dtype: float64

In [8]:
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')


In [9]:
outlier_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]

def count_outliers_3sigma(series: pd.Series) -> int:
    mu, sigma = series.mean(), series.std()
    return ((series < mu - 3 * sigma) | (series > mu + 3 * sigma)).sum()

pd.Series({c: count_outliers_3sigma(data[c].dropna()) for c in outlier_cols}, name="n_outliers_3sigma")

Temperature     0
Fuel_Price      0
CPI             0
Unemployment    5
Name: n_outliers_3sigma, dtype: int64

In [10]:
px.scatter_matrix(data,width=1000,height=1000).show()

In [11]:
# Mission Jedha : outliers = hors [mean - 3*std, mean + 3*std]
# sur Temperature, Fuel_Price, CPI, Unemployment
mask = pd.Series(True, index=data.index)
for col in outlier_cols:
    mu, sigma = data[col].mean(), data[col].std()
    mask &= data[col].between(mu - 3 * sigma, mu + 3 * sigma) | data[col].isna()

n_before = len(data)
data = data.loc[mask].copy()
print(f"Lignes avant outliers : {n_before} | après : {len(data)} | retirées : {n_before - len(data)}")

px.scatter_matrix(
    data,
    dimensions=outlier_cols + ["Weekly_Sales"],
    width=1000,
    height=1000,
).show()

Lignes avant outliers : 150 | après : 145 | retirées : 5


In [12]:
data['year'] = data.loc[:,'Date'].dt.year
data['month'] = data.loc[:,'Date'].dt.month
data['day'] = data.loc[:,'Date'].dt.day
data['day_of_week']=data.loc[:,'Date'].dt.dayofweek

In [13]:
data=data.loc[~data['Weekly_Sales'].isnull()]
(data.isnull().sum()/data.shape[0]*100).sort_values(ascending=False)

Date            13.740458
month           13.740458
day             13.740458
day_of_week     13.740458
year            13.740458
Unemployment    10.687023
Temperature     10.687023
Fuel_Price       9.160305
Holiday_Flag     8.396947
CPI              8.396947
Weekly_Sales     0.000000
Store            0.000000
dtype: float64

# Preprocessing

In [14]:
# data=data.dropna(subset=['month'])

In [15]:
X=data.drop(columns=['Weekly_Sales','Date'])
y=data['Weekly_Sales']

In [16]:
categorical_features=['Store','Holiday_Flag']
numeric_features= [c for c in X.columns if c not in categorical_features]

print('categorical_features :',categorical_features)
print('numeric_features :',numeric_features)


categorical_features : ['Store', 'Holiday_Flag']
numeric_features : ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'year', 'month', 'day', 'day_of_week']


In [17]:
X.isnull().sum()/len(X)*100

Store            0.000000
Holiday_Flag     8.396947
Temperature     10.687023
Fuel_Price       9.160305
CPI              8.396947
Unemployment    10.687023
year            13.740458
month           13.740458
day             13.740458
day_of_week     13.740458
dtype: float64

In [18]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OHE", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)


In [21]:
X_train[:1]

array([[ 7.88231923e-01,  1.05457686e+00,  8.91315877e-01,
         5.51919517e-01,  3.00252338e-13,  0.00000000e+00,
        -4.79847788e-16,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [22]:
X_test[:1]

array([[ 0.        ,  1.64269132,  0.13312602,  1.18204486,  0.1632108 ,
        -0.52926645, -1.37948558,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ]])

In [23]:
model=LinearRegression()
model.fit(X_train,Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](27,)","[-60993.5 ,-15278.68, 5777.05,...,-86576.8 ,462673.1 , 46227.46]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.6e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,27
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(26)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](27,)","[14.02,12.43,11.56,..., 0.76, 0.35, 0. ]"


In [24]:
print('Train Score :',model.score(X_train,Y_train))
print('Test Score :',model.score(X_test,Y_test))

Train Score : 0.9771347825598194
Test Score : 0.8908899782260363


In [25]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient,Intercept
19,x0_14.0,5.798212e+05,1.600078e+06
10,x0_4.0,4.875980e+05,1.600078e+06
25,x0_20.0,4.626731e+05,1.600078e+06
18,x0_13.0,3.718377e+05,1.600078e+06
8,x0_2.0,3.577429e+05,1.600078e+06
16,x0_10.0,3.303020e+05,1.600078e+06
5,month,7.036479e+04,1.600078e+06
26,x1_1.0,4.622746e+04,1.600078e+06
2,CPI,5.777049e+03,1.600078e+06
7,day_of_week,0.000000e+00,1.600078e+06


In [26]:
# pd.DataFrame({'features':numeric_features+categorical_features,'Coefficient':model.coef_,'Intercept':model.intercept_}).sort_values(by='Coefficient',ascending=False)

In [27]:
model=Ridge()

params={'alpha':list(np.linspace(0,10,100))}
gridsearch=GridSearchCV(model,param_grid=params,cv=10)

In [28]:
gridsearch.fit(X_train,Y_train)
print('Train Score :',gridsearch.score(X_train,Y_train))
print('Test Score :',gridsearch.score(X_test,Y_test))
print('Best Params :',gridsearch.best_params_)
print('Best Score :',gridsearch.best_score_)
print('Best Estimator :',gridsearch.best_estimator_)


Train Score : 0.9757433929515844
Test Score : 0.8953395430728919
Best Params : {'alpha': np.float64(0.10101010101010101)}
Best Score : 0.9371770720568181
Best Estimator : Ridge(alpha=np.float64(0.10101010101010101))


In [29]:
features_names=numeric_features+list(preprocessor.named_transformers_['cat'].named_steps['OHE'].get_feature_names_out())


pd.DataFrame({'features':features_names,'Coefficient':gridsearch.best_estimator_.coef_}).sort_values(by='Coefficient',ascending=False)

,features,Coefficient
19,x0_14.0,6.323429e+05
10,x0_4.0,6.258141e+05
25,x0_20.0,5.257505e+05
18,x0_13.0,4.798820e+05
8,x0_2.0,4.274791e+05
16,x0_10.0,4.036331e+05
12,x0_6.0,9.031253e+04
5,month,7.062359e+04
26,x1_1.0,5.876949e+04
2,CPI,2.223044e+04


# Selection des 3 meilleurs colonnes et remodelisation

In [30]:
X=data.loc[:,['Store','CPI','month']]
y=data['Weekly_Sales']

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [31]:
categorical_features=['Store']
numeric_features= ['CPI','month']

In [32]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OHE", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [33]:
data.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,day_of_week
0,6.0,2011-02-18,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13.0,2011-03-25,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
3,11.0,NaT,1244390.03,0.0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN
4,6.0,2010-05-28,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0
5,4.0,2010-05-28,1857533.70,0.0,NaN,2.756,126.160226,7.896,2010.0,5.0,28.0,4.0


In [34]:
from sklearn.model_selection import GridSearchCV, KFold

pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", Lasso(max_iter=10000, random_state=42)),
])


param_grid = {"model__alpha": np.linspace(0.0001, 10, 100)}
cv = KFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, Y_train)

print("Best params:", grid.best_params_)
print("CV best R² :", grid.best_score_)
print("Test R²    :", grid.score(X_test, Y_test))

Best params: {'model__alpha': np.float64(10.0)}
CV best R² : 0.9353728166517312
Test R²    : 0.8915352260838665


In [35]:
best_pipe = grid.best_estimator_
feature_names = best_pipe.named_steps["prep"].get_feature_names_out()
coefs = best_pipe.named_steps["model"].coef_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
      .sort_values("coef", ascending=False)
)
coef_df.head(20)

,feature,coef
13,cat__Store_14.0,5.970785e+05
4,cat__Store_4.0,5.844130e+05
19,cat__Store_20.0,5.576099e+05
2,cat__Store_2.0,4.160633e+05
12,cat__Store_13.0,4.018288e+05
10,cat__Store_10.0,1.858780e+05
1,num__month,7.379935e+04
6,cat__Store_6.0,7.358891e+04
0,num__CPI,-1.614541e+04
11,cat__Store_11.0,-7.067072e+04


# Conclusion

## Synthèse

L'analyse des ventes hebdomadaires Walmart couvre les 3 parties de la mission Jedha :

### 1. EDA & preprocessing
- Drop des lignes où Weekly_Sales est manquant (pas d'imputation sur la cible).
- Features date : year, month, day, day_of_week.
- Outliers retirés via la règle **±3σ** sur Temperature, Fuel_Price, CPI, Unemployment.
- Pipeline sklearn : imputation + StandardScaler (num) / imputation + OneHotEncoder (cat : Store, Holiday_Flag).

### 2. Baseline — LinearRegression
- Métrique : **R²**.
- Interprétation des coefficients : les magasins (Store), le mois et le CPI ressortent comme très influents.

### 3. Régularisation — Ridge & Lasso
- **Ridge** + GridSearchCV pour limiter le surapprentissage.
- **Lasso** (bonus) pour sélection de variables ; remodelisation sur les features les plus fortes (Store, CPI, month).

### Recommandation métier
Les ventes dépendent surtout du **magasin**, de la **saisonnalité (mois)** et du **CPI**. Un modèle régularisé est préférable pour planifier les campagnes marketing avec des prédictions plus stables.